Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Train

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]
PROTOCOL1_TRAIN_FILES = [
    ("1", "1_Augmented"), ("1", "2_Augmented"),
    ("2", "1_Augmented"), ("2", "2_Augmented"),
    ("3", "1_Augmented"), ("3", "2_Augmented"),
    ("4", "1_Augmented"), ("4", "2_Augmented")
]

train_data = []
train_labels = []

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print("🔧 Denoising image...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === LOAD TRAINING DATA FOR STRATEGY 2 ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="📥 Loading Protocol 2 - Strategy 2 Training"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_NUMS:
        for idx, (img_num, suffix) in enumerate(PROTOCOL1_TRAIN_FILES, start=1):
            fname = f"{subj}_{finger}_{img_num}_{suffix}.png"
            img_path = os.path.join(subject_path, fname)
            print(f"\n📁 Subject {subj} - Finger {finger} - Sample {idx:02d}")
            print(f"🖼️ Loading: {img_path}")

            if not os.path.exists(img_path):
                print(f"❌ File not found: {img_path}")
                continue

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, IMAGE_SIZE)
            img_denoised = apply_denoising(img, h=10)
            img_eq = exposure.equalize_hist(img_denoised)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            train_data.append(img_norm.flatten())
            label = os.path.splitext(fname)[0]  # keep same label as filename
            train_labels.append(label)
            print(f"✅ Training sample saved: {label}")

train_data = np.array(train_data)
train_labels = np.array(train_labels)

print("\n📊 ✅ Final Train Data Loaded")
print(f"   ➤ Total Samples: {train_data.shape[0]}")
print(f"   ➤ Feature Vector Length: {train_data.shape[1]}")
print(f"   ➤ Labels: {train_labels[:5]}")

# === PCA-Like 2DPCA SIMULATION FOR STRATEGY 2 ===
def compute_2dpca_1d(flat_data, num_components):
    print("\n⚙️ Simulating 2DPCA on flattened features...")
    mean_vector = np.mean(flat_data, axis=0)
    centered = flat_data - mean_vector
    cov = np.cov(centered, rowvar=False)
    eig_vals, eig_vecs = np.linalg.eigh(cov)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    return eig_vecs, mean_vector

# === PROJECT TO 2DPCA-LIKE SPACE ===
def project_flat_data(flat_data, eig_vecs, mean_vector):
    centered = flat_data - mean_vector
    return centered @ eig_vecs

num_components = 47
W, mean_vector = compute_2dpca_1d(train_data, num_components)
train_data_pca = project_flat_data(train_data, W, mean_vector)

print("\n✅ Projected Train Data Shape:", train_data_pca.shape)
print("📌 First Few Labels:", train_labels[:5])


Test

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]
PROTOCOL1_TEST_FILES = [
    ("1", "3_Augmented"),("2", "3_Augmented"), ("3", "3_Augmented"),("4", "3_Augmented"),
    ("1", ""), ("2", ""), ("3", ""), ("4", "")
]

test_data = []
test_labels = []

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    return cv2.fastNlMeansDenoising(image, h=h)

# === FUNCTION TO LOAD & PROCESS TEST IMAGES ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="🧪 Loading Protocol 2 - Strategy 2 Test"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_NUMS:
        for (img_num, suffix) in PROTOCOL1_TEST_FILES:
            if suffix:
                fname = f"{subj}_{finger}_{img_num}_{suffix}.png"
                label = f"{subj}_{finger}_{img_num}_{suffix}"
            else:
                fname = f"{subj}_{finger}_{img_num}.png"
                label = f"{subj}_{finger}_{img_num}_orig"

            img_path = os.path.join(subject_path, fname)
            print(f"\n🖼️ Loading: {img_path}")

            if not os.path.exists(img_path):
                print(f"❌ File not found: {img_path}")
                continue

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, IMAGE_SIZE)
            img_denoised = apply_denoising(img, h=10)
            img_eq = exposure.equalize_hist(img_denoised)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            test_data.append(img_norm.flatten())
            test_labels.append(label)
            print(f"✅ Test sample saved: {label}")

# === CONVERT TO NUMPY ARRAYS ===
test_data = np.array(test_data)
test_labels = np.array(test_labels)

print("\n📊 ✅ Final Test Data Loaded")
print(f"   ➤ Total Test Samples: {test_data.shape[0]}")
print(f"   ➤ Feature Vector Length: {test_data.shape[1]}")
print(f"   ➤ First Test Label: {test_labels[0] if len(test_labels) > 0 else 'None'}")

# === PROJECT TEST DATA ===
centered_test = test_data - mean_vector
proj_test_data = centered_test @ W
print("\n📐 Projected Test Data Shape:", proj_test_data.shape)


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(proj_test_data)

print("\n📤 Matching test samples using 2DPCA features (Strategy 2 — Protocol 2)...")

# === Step 1: Compare each test sample to all training samples ===
for i in range(total_tests):
    test_vector = proj_test_data[i]
    true_label = test_labels[i]  # e.g., "0032_3_4_3_Augmented"

    # 📏 Compute Manhattan distances to all training vectors
    distances = np.sum(np.abs(train_data_pca - test_vector), axis=1)

    # 🏆 Nearest neighbor index
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]

    # 🎯 Extract subject ID and finger number
    true_subject, true_finger = true_label.split("_")[0], true_label.split("_")[1]
    pred_subject, pred_finger = predicted_label.split("_")[0], predicted_label.split("_")[1]

    # ✅ Match check
    if pred_subject == true_subject and pred_finger == true_finger:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"🔍 Test {i+1:03d}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# === Final Accuracy Report ===
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Finger-wise Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")
